# Jetson Nano: Continuous Camera Display + Pose Overlay (CPU-Safe)

This notebook contains the stable CPU demo for pose estimation on Jetson Nano.

**Prerequisites:**
- Place these files in the same folder as this notebook:
  - `human_pose.json`
  - `resnet18_baseline_att_224x224_A_epoch_249.pth`

**Run this cell to start the demo:**

In [ ]:
# ===== Jetson Nano: Continuous camera display + pose overlay (CPU-safe) =====
# Place in: notebooks/live_demo_cpu_stable.ipynb
# Needs in SAME folder:
#   - human_pose.json
#   - resnet18_baseline_att_224x224_A_epoch_249.pth

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # disable CUDA (stability)

import cv2, time, json, threading
import torch
import ipywidgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg

import trt_pose.coco
import trt_pose.models
from trt_pose.parse_objects import ParseObjects
from trt_pose.draw_objects import DrawObjects

torch.set_num_threads(1)

# ---- Tunables ----
CAM_W, CAM_H, CAM_FPS = 320, 240, 30
MODEL_W, MODEL_H = 224, 224
DISPLAY_W, DISPLAY_H = 700, 700      # make smaller if it stutters (ex 480)
INFER_EVERY = 3                      # run AI every N frames
TARGET_UI_FPS = 15                   # widget refresh rate
# ------------------

# Camera (MJPG pipeline)
GST = (
    f"v4l2src device=/dev/video0 ! "
    f"image/jpeg,width={CAM_W},height={CAM_H},framerate={CAM_FPS}/1 ! "
    "jpegdec ! videoconvert ! "
    "video/x-raw,format=BGR ! appsink drop=1 sync=false max-buffers=1"
)
cap = cv2.VideoCapture(GST, cv2.CAP_GSTREAMER)
print("camera opened:", cap.isOpened())

# Widget display
image_w = widgets.Image(format="jpeg", width=DISPLAY_W, height=DISPLAY_H)
display(image_w)

# Preprocess
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def preprocess(image_bgr_224):
    img = cv2.cvtColor(image_bgr_224, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(img).permute(2,0,1).float() / 255.0
    t = t.sub(mean).div(std)
    return t.unsqueeze(0)

# Load topology/model
with open("human_pose.json", "r") as f:
    human_pose = json.load(f)

topology = trt_pose.coco.coco_category_to_topology(human_pose)
num_parts = len(human_pose["keypoints"])
num_links = len(human_pose["skeleton"])

model = trt_pose.models.resnet18_baseline_att(num_parts, 2 * num_links).eval()
MODEL_WEIGHTS = "resnet18_baseline_att_224x224_A_epoch_249.pth"
model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location="cpu"))

parse_objects = ParseObjects(topology)
draw_objects  = DrawObjects(topology)

print("✅ model loaded (CPU)")

# Continuous loop
pose_running = True
_last_counts = None
_last_objects = None
_last_peaks = None

def pose_loop():
    global pose_running, _last_counts, _last_objects, _last_peaks

    i = 0
    last_ui = 0.0
    ui_period = 1.0 / max(1, TARGET_UI_FPS)

    while pose_running:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.01)
            continue

        frame224 = cv2.resize(frame, (MODEL_W, MODEL_H))

        # Inference every N frames
        if i % INFER_EVERY == 0:
            data = preprocess(frame224)
            with torch.no_grad():
                cmap, paf = model(data)
            _last_counts, _last_objects, _last_peaks = parse_objects(cmap, paf)

        # Draw last pose every frame
        if _last_counts is not None:
            draw_objects(frame224, _last_counts, _last_objects, _last_peaks)

        # UI update
        now = time.time()
        if now - last_ui >= ui_period:
            display_frame = cv2.resize(frame224, (DISPLAY_W, DISPLAY_H), interpolation=cv2.INTER_NEAREST)
            image_w.value = bgr8_to_jpeg(display_frame[:, ::-1, :])
            last_ui = now

        i += 1

t = threading.Thread(target=pose_loop, daemon=True)
t.start()
print("✅ Running continuously. Stop with stop_pose()")

def stop_pose():
    global pose_running
    pose_running = False
    time.sleep(0.2)
    cap.release()
    print("🛑 stopped + camera released")

## Stop the Demo

Run this cell to stop the pose detection and release the camera:

In [ ]:
# Stop the pose detection
stop_pose()